# Final Edge TTS Audio Generation

Regenerates approved Arabic letter and number audio from **exact Edge TTS text** and **exact cut rules** traced from source notebooks.

Reference stems (not copied): `generated_audio/edge_tts_mapping_v2/chosen/` and `generated_audio/edge_tts_numbers_mapping_v1/chosen/`.

Output: `audio/letters/`, `audio/numbers/`, temp in `audio/_temp/`.


## Imports and Paths

Install dependencies, set paths, create output folders.


In [1]:
import asyncio
import shutil
import subprocess
import sys
import time
from pathlib import Path

def _ensure(pkg: str, import_name: str | None = None):
    name = import_name or pkg
    try:
        __import__(name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        __import__(name)

_ensure("edge-tts", "edge_tts")
_ensure("nest_asyncio")
_ensure("pydub")
_ensure("IPython", "IPython")

import nest_asyncio
nest_asyncio.apply()

BASE_DIR = Path.cwd().resolve()
AUDIO_ROOT = BASE_DIR / "audio"
DIR_LETTERS = AUDIO_ROOT / "letters"
DIR_NUMBERS = AUDIO_ROOT / "numbers"
DIR_TEMP = AUDIO_ROOT / "_temp"

REF_LETTERS_CHOSEN = BASE_DIR / "generated_audio" / "edge_tts_mapping_v2" / "chosen"
REF_NUMBERS_CHOSEN = BASE_DIR / "generated_audio" / "edge_tts_numbers_mapping_v1" / "chosen"

for d in (DIR_LETTERS, DIR_NUMBERS, DIR_TEMP):
    d.mkdir(parents=True, exist_ok=True)

print("cwd:", BASE_DIR)
print("output letters:", DIR_LETTERS)
print("output numbers:", DIR_NUMBERS)
print("reference letters chosen:", REF_LETTERS_CHOSEN)
print("reference numbers chosen:", REF_NUMBERS_CHOSEN)


cwd: C:\Users\sondo\Desktop\Voice Model Stuff
output letters: C:\Users\sondo\Desktop\Voice Model Stuff\audio\letters
output numbers: C:\Users\sondo\Desktop\Voice Model Stuff\audio\numbers
reference letters chosen: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\chosen
reference numbers chosen: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers_mapping_v1\chosen


## Edge TTS Voice Setup

Select `ar-EG-SalmaNeural`, falling back to `ar-EG-ShakirNeural`.


In [2]:
import edge_tts

PREFERRED_VOICES = ["ar-EG-SalmaNeural", "ar-EG-ShakirNeural"]
_ALL_VOICES = None

async def _list_voices():
    global _ALL_VOICES
    if _ALL_VOICES is None:
        _ALL_VOICES = await edge_tts.list_voices()
    return _ALL_VOICES

async def select_edge_voice() -> str:
    voices = await _list_voices()
    by_short = {v["ShortName"]: v for v in voices}
    for pref in PREFERRED_VOICES:
        if pref in by_short:
            return pref
    ar_names = sorted(
        v["ShortName"]
        for v in voices
        if str(v.get("Locale", "")).lower().startswith("ar")
    )
    if not ar_names:
        raise RuntimeError("No Arabic Edge TTS voice found")
    return ar_names[0]

EDGE_VOICE = asyncio.get_event_loop().run_until_complete(select_edge_voice())
print("Selected Edge TTS voice:", EDGE_VOICE)


Selected Edge TTS voice: ar-EG-SalmaNeural


## Approved Provenance (from source notebooks)

Each record ties a final letter/number to the chosen stem, source notebook section, TTS text, and cut rule.


In [3]:
APPROVED_LETTER_RECORDS = [
    {"letter": "ا", "chosen_filename": "letter_ا_alif.mp3", "source_notebook": "edge7_tts_full_arabic_voice_test.ipynb", "source_section": "CHOSEN_MAPPINGS", "tts_text": "أَلِف.", "cut_rule": None},
    {"letter": "ب", "chosen_filename": "(not in chosen folder)", "source_notebook": "edge4_tts_full_arabic_voice_test.ipynb", "source_section": "COMPARISON_VARIANTS → comp_beh_ب_ب_ه", "tts_text": "بِه.", "cut_rule": {'duration_seconds': 0.6, 'trim_leading_silence': True}},
    {"letter": "ت", "chosen_filename": "letter_ت_teh.mp3", "source_notebook": "edge7_tts_full_arabic_voice_test.ipynb", "source_section": "CHOSEN_MAPPINGS", "tts_text": "تِه.", "cut_rule": None},
    {"letter": "ث", "chosen_filename": "comp_seh_ث_ث_ه.mp3", "source_notebook": "edge7_tts_full_arabic_voice_test.ipynb", "source_section": "CHOSEN_MAPPINGS", "tts_text": "ثِه.", "cut_rule": None},
    {"letter": "ج", "chosen_filename": "letter_ج_geem.mp3", "source_notebook": "edge7_tts_full_arabic_voice_test.ipynb", "source_section": "CHOSEN_MAPPINGS", "tts_text": "جِيم.", "cut_rule": None},
    {"letter": "ح", "chosen_filename": "letter_ح_ha.mp3", "source_notebook": "edge7_tts_full_arabic_voice_test.ipynb", "source_section": "CHOSEN_MAPPINGS", "tts_text": "حَا.", "cut_rule": None},
    {"letter": "خ", "chosen_filename": "letter_خ_kha.mp3", "source_notebook": "edge7_tts_full_arabic_voice_test.ipynb", "source_section": "CHOSEN_MAPPINGS", "tts_text": "خَه.", "cut_rule": None},
    {"letter": "د", "chosen_filename": "comp_dal_د_دال_bang.mp3", "source_notebook": "edge5_tts_full_arabic_voice_test.ipynb", "source_section": "COMPARISON_VARIANTS (دالْ! / bang)", "tts_text": "دال!", "cut_rule": None},
    {"letter": "ذ", "chosen_filename": "comp_zal_ذ_ذال_bang.mp3", "source_notebook": "edge5_tts_full_arabic_voice_test.ipynb", "source_section": "COMPARISON_VARIANTS (ذالْ! / bang)", "tts_text": "ذال!", "cut_rule": None},
    {"letter": "ر", "chosen_filename": "letter_ر_ra.mp3", "source_notebook": "edge7_tts_full_arabic_voice_test.ipynb", "source_section": "CHOSEN_MAPPINGS", "tts_text": "رَا.", "cut_rule": None},
    {"letter": "ز", "chosen_filename": "comp_zay_ز_zeen_latin_long.mp3", "source_notebook": "edge5_tts_full_arabic_voice_test.ipynb", "source_section": "COMPARISON_VARIANTS (zeen / latin_long)", "tts_text": "zeen.", "cut_rule": None},
    {"letter": "س", "chosen_filename": "comp_seen_س_seen_split_cut.mp3", "source_notebook": "edge7_tts_full_arabic_voice_test.ipynb", "source_section": "SEEN_COMPARISON_VARIANTS (seen_split) + cut like num_02 §6b", "tts_text": "س ي ن.", "cut_rule": {'duration_seconds': 0.6, 'trim_leading_silence': True}},
    {"letter": "ش", "chosen_filename": "letter_ش_sheen.mp3", "source_notebook": "edge7_tts_full_arabic_voice_test.ipynb", "source_section": "CHOSEN_MAPPINGS", "tts_text": "شِين.", "cut_rule": None},
    {"letter": "ص", "chosen_filename": "comp_sad_ص_saad_long_sukoon.mp3", "source_notebook": "edge7_tts_full_arabic_voice_test.ipynb", "source_section": "FINAL_COMPARISON_VARIANTS (round 6 promote)", "tts_text": "صَاادْ.", "cut_rule": None},
    {"letter": "ض", "chosen_filename": "comp_dad_ض_daad_long_sukoon.mp3", "source_notebook": "edge7_tts_full_arabic_voice_test.ipynb", "source_section": "FINAL_COMPARISON_VARIANTS (round 6 promote)", "tts_text": "ضَاادْ.", "cut_rule": None},
    {"letter": "ط", "chosen_filename": "comp_ta_ط_ط_ه.mp3", "source_notebook": "edge7_tts_full_arabic_voice_test.ipynb", "source_section": "CHOSEN_MAPPINGS", "tts_text": "طَه.", "cut_rule": None},
    {"letter": "ظ", "chosen_filename": "letter_ظ_dha.mp3", "source_notebook": "edge7_tts_full_arabic_voice_test.ipynb", "source_section": "CHOSEN_MAPPINGS", "tts_text": "ظَه.", "cut_rule": None},
    {"letter": "ع", "chosen_filename": "letter_ع_ein.mp3", "source_notebook": "edge7_tts_full_arabic_voice_test.ipynb", "source_section": "CHOSEN_MAPPINGS", "tts_text": "عِين.", "cut_rule": None},
    {"letter": "غ", "chosen_filename": "letter_غ_ghein.mp3", "source_notebook": "edge7_tts_full_arabic_voice_test.ipynb", "source_section": "CHOSEN_MAPPINGS", "tts_text": "غِين.", "cut_rule": None},
    {"letter": "ف", "chosen_filename": "comp_faa_ف_ف_ه.mp3", "source_notebook": "edge7_tts_full_arabic_voice_test.ipynb", "source_section": "CHOSEN_MAPPINGS", "tts_text": "فِه.", "cut_rule": None},
    {"letter": "ق", "chosen_filename": "comp_qaf_ق_qaf_split_cut_v2.mp3", "source_notebook": "edge7_tts_full_arabic_voice_test.ipynb", "source_section": "§6c cut trailing + §6c2 keep first 0.61s of v1 cut", "tts_text": "قا فْ.", "cut_rule": {'qaf_two_stage': True, 'duration_seconds': 0.61}},
    {"letter": "ك", "chosen_filename": "comp_kaf_ك_كاف_sukoon.mp3", "source_notebook": "edge5_tts_full_arabic_voice_test.ipynb", "source_section": "COMPARISON_VARIANTS (كافْ / sukoon)", "tts_text": "كافْ.", "cut_rule": None},
    {"letter": "ل", "chosen_filename": "comp_lam_ل_lam_latin.mp3", "source_notebook": "edge5_tts_full_arabic_voice_test.ipynb", "source_section": "COMPARISON_VARIANTS (lam / latin)", "tts_text": "lam.", "cut_rule": None},
    {"letter": "م", "chosen_filename": "letter_م_meem.mp3", "source_notebook": "edge7_tts_full_arabic_voice_test.ipynb", "source_section": "CHOSEN_MAPPINGS", "tts_text": "مِيم.", "cut_rule": None},
    {"letter": "ن", "chosen_filename": "letter_ن_noon.mp3", "source_notebook": "edge7_tts_full_arabic_voice_test.ipynb", "source_section": "CHOSEN_MAPPINGS", "tts_text": "نُون.", "cut_rule": None},
    {"letter": "ه", "chosen_filename": "comp_ha_ه_ه_ه.mp3", "source_notebook": "edge7_tts_full_arabic_voice_test.ipynb", "source_section": "CHOSEN_MAPPINGS", "tts_text": "هِه.", "cut_rule": None},
    {"letter": "و", "chosen_filename": "comp_waw_و_واو_diac.mp3", "source_notebook": "edge5_tts_full_arabic_voice_test.ipynb", "source_section": "COMPARISON_VARIANTS (وَاْو / diac)", "tts_text": "وَاْو.", "cut_rule": None},
    {"letter": "ي", "chosen_filename": "letter_ي_yeh.mp3", "source_notebook": "edge7_tts_full_arabic_voice_test.ipynb", "source_section": "CHOSEN_MAPPINGS", "tts_text": "يِه.", "cut_rule": None},
]

APPROVED_NUMBER_RECORDS = [
    {"number": 0, "chosen_filename": "num_00_v02.mp3", "source_notebook": "edge_tts_number_pronunciation_test.ipynb", "source_section": "CHOSEN_V1_STEMS (variant v02)", "tts_text": "صِفِر.", "cut_rule": None},
    {"number": 1, "chosen_filename": "num_01.mp3", "source_notebook": "edge7_tts_full_arabic_voice_test.ipynb", "source_section": "NUMBER_PRON → edge_tts_numbers", "tts_text": "وَاحِد.", "cut_rule": None},
    {"number": 2, "chosen_filename": "num_02_atneen_bang_cut_0_59.mp3", "source_notebook": "edge_tts_number_pronunciation_test.ipynb", "source_section": "§6b cut num_02_atneen_bang", "tts_text": "اتنين!", "cut_rule": {'duration_seconds': 0.59, 'trim_leading_silence': True}},
    {"number": 3, "chosen_filename": "num_03.mp3", "source_notebook": "edge7_tts_full_arabic_voice_test.ipynb", "source_section": "NUMBER_PRON → edge_tts_numbers", "tts_text": "تَلَاتَه.", "cut_rule": None},
    {"number": 4, "chosen_filename": "num_04.mp3", "source_notebook": "edge7_tts_full_arabic_voice_test.ipynb", "source_section": "NUMBER_PRON → edge_tts_numbers", "tts_text": "أَرْبَعَه.", "cut_rule": None},
    {"number": 5, "chosen_filename": "num_05.mp3", "source_notebook": "edge7_tts_full_arabic_voice_test.ipynb", "source_section": "NUMBER_PRON → edge_tts_numbers", "tts_text": "خَمْسَه.", "cut_rule": None},
    {"number": 6, "chosen_filename": "num_06.mp3", "source_notebook": "edge7_tts_full_arabic_voice_test.ipynb", "source_section": "NUMBER_PRON → edge_tts_numbers", "tts_text": "سِتَّه.", "cut_rule": None},
    {"number": 7, "chosen_filename": "num_07.mp3", "source_notebook": "edge7_tts_full_arabic_voice_test.ipynb", "source_section": "NUMBER_PRON → edge_tts_numbers", "tts_text": "سَبْعَه.", "cut_rule": None},
    {"number": 8, "chosen_filename": "num_08.mp3", "source_notebook": "edge7_tts_full_arabic_voice_test.ipynb", "source_section": "NUMBER_PRON → edge_tts_numbers", "tts_text": "تَمَانْيَه.", "cut_rule": None},
    {"number": 9, "chosen_filename": "num_09.mp3", "source_notebook": "edge7_tts_full_arabic_voice_test.ipynb", "source_section": "NUMBER_PRON → edge_tts_numbers", "tts_text": "تِسْعَه.", "cut_rule": None},
    {"number": 10, "chosen_filename": "num_10_asharah_sukoon.mp3", "source_notebook": "edge_tts_number_pronunciation_test.ipynb", "source_section": "ROUND2_COMPARISON promote → chosen", "tts_text": "عشرةْ.", "cut_rule": None},
]


## Final Approved Letter Mappings

Built from `APPROVED_LETTER_RECORDS`.


In [4]:
FINAL_LETTER_MAPPINGS = {r["letter"]: r["tts_text"] for r in APPROVED_LETTER_RECORDS}

LETTER_CUT_RULES = {
    r["letter"]: r["cut_rule"]
    for r in APPROVED_LETTER_RECORDS
    if r["cut_rule"] is not None
}

print("Letter mappings:", len(FINAL_LETTER_MAPPINGS))
for ch in FINAL_LETTER_MAPPINGS:
    print(f"  {ch} -> {FINAL_LETTER_MAPPINGS[ch]!r}")
print("Letter cut rules:", LETTER_CUT_RULES)


Letter mappings: 28
  ا -> 'أَلِف.'
  ب -> 'بِه.'
  ت -> 'تِه.'
  ث -> 'ثِه.'
  ج -> 'جِيم.'
  ح -> 'حَا.'
  خ -> 'خَه.'
  د -> 'دال!'
  ذ -> 'ذال!'
  ر -> 'رَا.'
  ز -> 'zeen.'
  س -> 'س ي ن.'
  ش -> 'شِين.'
  ص -> 'صَاادْ.'
  ض -> 'ضَاادْ.'
  ط -> 'طَه.'
  ظ -> 'ظَه.'
  ع -> 'عِين.'
  غ -> 'غِين.'
  ف -> 'فِه.'
  ق -> 'قا فْ.'
  ك -> 'كافْ.'
  ل -> 'lam.'
  م -> 'مِيم.'
  ن -> 'نُون.'
  ه -> 'هِه.'
  و -> 'وَاْو.'
  ي -> 'يِه.'
Letter cut rules: {'ب': {'duration_seconds': 0.6, 'trim_leading_silence': True}, 'س': {'duration_seconds': 0.6, 'trim_leading_silence': True}, 'ق': {'qaf_two_stage': True, 'duration_seconds': 0.61}}


## Final Approved Number Mappings

Built from `APPROVED_NUMBER_RECORDS`.


In [5]:
FINAL_NUMBER_MAPPINGS = {r["number"]: r["tts_text"] for r in APPROVED_NUMBER_RECORDS}

NUMBER_CUT_RULES = {
    r["number"]: r["cut_rule"]
    for r in APPROVED_NUMBER_RECORDS
    if r["cut_rule"] is not None
}

print("Number mappings:", len(FINAL_NUMBER_MAPPINGS))
for n in sorted(FINAL_NUMBER_MAPPINGS):
    print(f"  {n} -> {FINAL_NUMBER_MAPPINGS[n]!r}")
print("Number cut rules:", NUMBER_CUT_RULES)


Number mappings: 11
  0 -> 'صِفِر.'
  1 -> 'وَاحِد.'
  2 -> 'اتنين!'
  3 -> 'تَلَاتَه.'
  4 -> 'أَرْبَعَه.'
  5 -> 'خَمْسَه.'
  6 -> 'سِتَّه.'
  7 -> 'سَبْعَه.'
  8 -> 'تَمَانْيَه.'
  9 -> 'تِسْعَه.'
  10 -> 'عشرةْ.'
Number cut rules: {2: {'duration_seconds': 0.59, 'trim_leading_silence': True}}


## Mapping Audit Printout

Print every approved mapping with chosen filename, source notebook, TTS text, cut rule, and final path.


In [6]:
print("=== Letter mapping audit ===\n")
for r in APPROVED_LETTER_RECORDS:
    ch = r["letter"]
    cut = r["cut_rule"]
    cut_s = "none" if cut is None else repr(cut)
    final_path = DIR_LETTERS / f"letter_{ch}.mp3"
    print(f"letter {ch}")
    print(f"  chosen filename: {r['chosen_filename']}")
    print(f"  source notebook: {r['source_notebook']}")
    print(f"  source section: {r['source_section']}")
    print(f"  Edge TTS text: {r['tts_text']!r}")
    print(f"  cut rule: {cut_s}")
    print(f"  final path: {final_path.resolve()}")
    print()

print("=== Number mapping audit ===\n")
for r in APPROVED_NUMBER_RECORDS:
    n = r["number"]
    cut = r["cut_rule"]
    cut_s = "none" if cut is None else repr(cut)
    final_path = DIR_NUMBERS / f"number_{n}.mp3"
    print(f"number {n}")
    print(f"  chosen filename: {r['chosen_filename']}")
    print(f"  source notebook: {r['source_notebook']}")
    print(f"  source section: {r['source_section']}")
    print(f"  Edge TTS text: {r['tts_text']!r}")
    print(f"  cut rule: {cut_s}")
    print(f"  final path: {final_path.resolve()}")
    print()


=== Letter mapping audit ===

letter ا
  chosen filename: letter_ا_alif.mp3
  source notebook: edge7_tts_full_arabic_voice_test.ipynb
  source section: CHOSEN_MAPPINGS
  Edge TTS text: 'أَلِف.'
  cut rule: none
  final path: C:\Users\sondo\Desktop\Voice Model Stuff\audio\letters\letter_ا.mp3

letter ب
  chosen filename: (not in chosen folder)
  source notebook: edge4_tts_full_arabic_voice_test.ipynb
  source section: COMPARISON_VARIANTS → comp_beh_ب_ب_ه
  Edge TTS text: 'بِه.'
  cut rule: {'duration_seconds': 0.6, 'trim_leading_silence': True}
  final path: C:\Users\sondo\Desktop\Voice Model Stuff\audio\letters\letter_ب.mp3

letter ت
  chosen filename: letter_ت_teh.mp3
  source notebook: edge7_tts_full_arabic_voice_test.ipynb
  source section: CHOSEN_MAPPINGS
  Edge TTS text: 'تِه.'
  cut rule: none
  final path: C:\Users\sondo\Desktop\Voice Model Stuff\audio\letters\letter_ت.mp3

letter ث
  chosen filename: comp_seh_ث_ث_ه.mp3
  source notebook: edge7_tts_full_arabic_voice_test.ipynb
 

## Synthesis Helpers

Edge TTS export and pydub cuts matching edge7 (ق two-stage) and edge_tts_number_pronunciation_test (leading trim + duration).


In [7]:
from pydub import AudioSegment
from pydub.silence import detect_nonsilent

async def synth_mp3(text: str, out_path: Path) -> float:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.perf_counter()
    comm = edge_tts.Communicate(text, EDGE_VOICE)
    await comm.save(str(out_path))
    return time.perf_counter() - t0

def trim_leading_silence(audio: AudioSegment) -> AudioSegment:
    nonsilent = detect_nonsilent(
        audio, min_silence_len=50, silence_thresh=audio.dBFS - 14, seek_step=10
    )
    if nonsilent:
        return audio[nonsilent[0][0] :]
    return audio

def trim_trailing_silence(audio: AudioSegment) -> AudioSegment:
    nonsilent = detect_nonsilent(
        audio, min_silence_len=80, silence_thresh=audio.dBFS - 14, seek_step=10
    )
    if nonsilent:
        end_ms = min(len(audio), nonsilent[-1][1] + 120)
        return audio[:end_ms]
    return audio

def cut_to_duration(
    src: Path,
    dest: Path,
    duration_seconds: float,
    *,
    trim_leading: bool = False,
    trim_trailing: bool = False,
) -> float:
    audio = AudioSegment.from_mp3(src)
    if trim_leading:
        audio = trim_leading_silence(audio)
    if trim_trailing:
        audio = trim_trailing_silence(audio)
    cut = audio[: int(duration_seconds * 1000)]
    cut.export(dest, format="mp3")
    return len(cut) / 1000.0

def cut_qaf_two_stage(raw_path: Path, dest: Path, duration_seconds: float = 0.61) -> float:
    """Match edge7 §6c + §6c2: trim trailing on full TTS, then keep first 0.61s."""
    audio = AudioSegment.from_mp3(raw_path)
    v1 = trim_trailing_silence(audio)
    v2 = v1[: int(duration_seconds * 1000)]
    v2.export(dest, format="mp3")
    return len(v2) / 1000.0

def apply_letter_cut(raw_path: Path, dest: Path, rule: dict) -> float:
    if rule.get("qaf_two_stage"):
        return cut_qaf_two_stage(raw_path, dest, rule["duration_seconds"])
    return cut_to_duration(
        raw_path,
        dest,
        rule["duration_seconds"],
        trim_leading=rule.get("trim_leading_silence", False),
        trim_trailing=rule.get("trim_trailing_silence", False),
    )

def run_async(coro):
    return asyncio.get_event_loop().run_until_complete(coro)


## Generate Final Letter Audio

Synthesize each letter; when a cut rule exists, save the **cut** clip.


In [8]:
LETTER_ORDER = list("ابتثجحخدذرزسشصضطظعغفقكلمنهوي")
assert set(LETTER_ORDER) == set(FINAL_LETTER_MAPPINGS)

letter_results = []

for letter in LETTER_ORDER:
    text = FINAL_LETTER_MAPPINGS[letter]
    meta = next(r for r in APPROVED_LETTER_RECORDS if r["letter"] == letter)
    final_path = DIR_LETTERS / f"letter_{letter}.mp3"
    cut_rule = LETTER_CUT_RULES.get(letter)

    raw_path = DIR_TEMP / f"letter_{letter}_raw.mp3"
    infer_s = run_async(synth_mp3(text, raw_path))

    if cut_rule:
        cut_path = DIR_TEMP / f"letter_{letter}_cut.mp3"
        dur = apply_letter_cut(raw_path, cut_path, cut_rule)
        shutil.copy2(cut_path, final_path)
        print(
            f"letter {letter} | chosen: {meta['chosen_filename']} | TTS: {text} | "
            f"cut: yes ({dur:.3f}s) | saved: {final_path}"
        )
    else:
        shutil.copy2(raw_path, final_path)
        dur = len(AudioSegment.from_mp3(final_path)) / 1000.0
        print(
            f"letter {letter} | chosen: {meta['chosen_filename']} | TTS: {text} | "
            f"cut: no | saved: {final_path}"
        )

    letter_results.append(
        {
            "letter": letter,
            "text": text,
            "cut": cut_rule is not None,
            "path": final_path.resolve(),
            "infer_s": infer_s,
            "duration_s": dur,
        }
    )

for p in DIR_TEMP.glob("letter_*"):
    p.unlink(missing_ok=True)

print(f"\nGenerated {len(letter_results)} letter MP3 files in {DIR_LETTERS}")


letter ا | chosen: letter_ا_alif.mp3 | TTS: أَلِف. | cut: no | saved: C:\Users\sondo\Desktop\Voice Model Stuff\audio\letters\letter_ا.mp3


letter ب | chosen: (not in chosen folder) | TTS: بِه. | cut: yes (0.600s) | saved: C:\Users\sondo\Desktop\Voice Model Stuff\audio\letters\letter_ب.mp3


letter ت | chosen: letter_ت_teh.mp3 | TTS: تِه. | cut: no | saved: C:\Users\sondo\Desktop\Voice Model Stuff\audio\letters\letter_ت.mp3


letter ث | chosen: comp_seh_ث_ث_ه.mp3 | TTS: ثِه. | cut: no | saved: C:\Users\sondo\Desktop\Voice Model Stuff\audio\letters\letter_ث.mp3


letter ج | chosen: letter_ج_geem.mp3 | TTS: جِيم. | cut: no | saved: C:\Users\sondo\Desktop\Voice Model Stuff\audio\letters\letter_ج.mp3


...

letter م | chosen: letter_م_meem.mp3 | TTS: مِيم. | cut: no | saved: C:\Users\sondo\Desktop\Voice Model Stuff\audio\letters\letter_م.mp3


letter ن | chosen: letter_ن_noon.mp3 | TTS: نُون. | cut: no | saved: C:\Users\sondo\Desktop\Voice Model Stuff\audio\letters\letter_ن.mp3


letter ه | chosen: comp_ha_ه_ه_ه.mp3 | TTS: هِه. | cut: no | saved: C:\Users\sondo\Desktop\Voice Model Stuff\audio\letters\letter_ه.mp3


letter و | chosen: comp_waw_و_واو_diac.mp3 | TTS: وَاْو. | cut: no | saved: C:\Users\sondo\Desktop\Voice Model Stuff\audio\letters\letter_و.mp3


letter ي | chosen: letter_ي_yeh.mp3 | TTS: يِه. | cut: no | saved: C:\Users\sondo\Desktop\Voice Model Stuff\audio\letters\letter_ي.mp3

Generated 28 letter MP3 files in C:\Users\sondo\Desktop\Voice Model Stuff\audio\letters


## Generate Final Number Audio

Synthesize each number; apply cut for 2 only.


In [9]:
number_results = []

for n in sorted(FINAL_NUMBER_MAPPINGS):
    text = FINAL_NUMBER_MAPPINGS[n]
    meta = next(r for r in APPROVED_NUMBER_RECORDS if r["number"] == n)
    final_path = DIR_NUMBERS / f"number_{n}.mp3"
    cut_rule = NUMBER_CUT_RULES.get(n)

    raw_path = DIR_TEMP / f"number_{n}_raw.mp3"
    infer_s = run_async(synth_mp3(text, raw_path))

    if cut_rule:
        cut_path = DIR_TEMP / f"number_{n}_cut.mp3"
        dur = cut_to_duration(
            raw_path,
            cut_path,
            cut_rule["duration_seconds"],
            trim_leading=cut_rule.get("trim_leading_silence", False),
            trim_trailing=cut_rule.get("trim_trailing_silence", False),
        )
        shutil.copy2(cut_path, final_path)
        print(
            f"number {n} | chosen: {meta['chosen_filename']} | TTS: {text} | "
            f"cut: yes ({dur:.3f}s) | saved: {final_path}"
        )
    else:
        shutil.copy2(raw_path, final_path)
        dur = len(AudioSegment.from_mp3(final_path)) / 1000.0
        print(
            f"number {n} | chosen: {meta['chosen_filename']} | TTS: {text} | "
            f"cut: no | saved: {final_path}"
        )

    number_results.append(
        {
            "number": n,
            "text": text,
            "cut": cut_rule is not None,
            "path": final_path.resolve(),
            "infer_s": infer_s,
            "duration_s": dur,
        }
    )

for p in DIR_TEMP.glob("number_*"):
    p.unlink(missing_ok=True)

print(f"\nGenerated {len(number_results)} number MP3 files in {DIR_NUMBERS}")


number 0 | chosen: num_00_v02.mp3 | TTS: صِفِر. | cut: no | saved: C:\Users\sondo\Desktop\Voice Model Stuff\audio\numbers\number_0.mp3


number 1 | chosen: num_01.mp3 | TTS: وَاحِد. | cut: no | saved: C:\Users\sondo\Desktop\Voice Model Stuff\audio\numbers\number_1.mp3


number 2 | chosen: num_02_atneen_bang_cut_0_59.mp3 | TTS: اتنين! | cut: yes (0.590s) | saved: C:\Users\sondo\Desktop\Voice Model Stuff\audio\numbers\number_2.mp3


number 3 | chosen: num_03.mp3 | TTS: تَلَاتَه. | cut: no | saved: C:\Users\sondo\Desktop\Voice Model Stuff\audio\numbers\number_3.mp3


number 4 | chosen: num_04.mp3 | TTS: أَرْبَعَه. | cut: no | saved: C:\Users\sondo\Desktop\Voice Model Stuff\audio\numbers\number_4.mp3


...

number 6 | chosen: num_06.mp3 | TTS: سِتَّه. | cut: no | saved: C:\Users\sondo\Desktop\Voice Model Stuff\audio\numbers\number_6.mp3


number 7 | chosen: num_07.mp3 | TTS: سَبْعَه. | cut: no | saved: C:\Users\sondo\Desktop\Voice Model Stuff\audio\numbers\number_7.mp3


number 8 | chosen: num_08.mp3 | TTS: تَمَانْيَه. | cut: no | saved: C:\Users\sondo\Desktop\Voice Model Stuff\audio\numbers\number_8.mp3


number 9 | chosen: num_09.mp3 | TTS: تِسْعَه. | cut: no | saved: C:\Users\sondo\Desktop\Voice Model Stuff\audio\numbers\number_9.mp3


number 10 | chosen: num_10_asharah_sukoon.mp3 | TTS: عشرةْ. | cut: no | saved: C:\Users\sondo\Desktop\Voice Model Stuff\audio\numbers\number_10.mp3

Generated 11 number MP3 files in C:\Users\sondo\Desktop\Voice Model Stuff\audio\numbers


## Verify Output Counts

Confirm 28 letter files and 11 number files exist.


In [10]:
expected_letters = [DIR_LETTERS / f"letter_{ch}.mp3" for ch in LETTER_ORDER]
expected_numbers = [DIR_NUMBERS / f"number_{n}.mp3" for n in range(11)]

letter_files = sorted(DIR_LETTERS.glob("letter_*.mp3"))
number_files = sorted(DIR_NUMBERS.glob("number_*.mp3"))

missing_letters = [p for p in expected_letters if not p.is_file()]
missing_numbers = [p for p in expected_numbers if not p.is_file()]

print("Letter MP3 count:", len(letter_files), "(expected 28)")
print("Number MP3 count:", len(number_files), "(expected 11)")
print("Missing letters:", missing_letters or "none")
print("Missing numbers:", missing_numbers or "none")
assert len(letter_files) == 28 and not missing_letters
assert len(number_files) == 11 and not missing_numbers
print("Verification OK")


Letter MP3 count: 28 (expected 28)
Number MP3 count: 11 (expected 11)
Missing letters: none
Missing numbers: none
Verification OK


## Playback Samples

Listen to regenerated clips including cut letters ب, ق, س and number 2.


In [11]:
from IPython.display import Audio, display

samples = [
    ("letter", DIR_LETTERS / "letter_ا.mp3"),
    ("letter ب (cut 0.6s)", DIR_LETTERS / "letter_ب.mp3"),
    ("letter (cut) ق", DIR_LETTERS / "letter_ق.mp3"),
    ("letter (cut) س", DIR_LETTERS / "letter_س.mp3"),
    ("number 0", DIR_NUMBERS / "number_0.mp3"),
    ("number (cut) 2", DIR_NUMBERS / "number_2.mp3"),
    ("number 10", DIR_NUMBERS / "number_10.mp3"),
]

for label, path in samples:
    print(label, "->", path.resolve())
    display(Audio(filename=str(path)))
    print()


letter -> C:\Users\sondo\Desktop\Voice Model Stuff\audio\letters\letter_ا.mp3



letter ب (cut 0.6s) -> C:\Users\sondo\Desktop\Voice Model Stuff\audio\letters\letter_ب.mp3



letter (cut) ق -> C:\Users\sondo\Desktop\Voice Model Stuff\audio\letters\letter_ق.mp3



letter (cut) س -> C:\Users\sondo\Desktop\Voice Model Stuff\audio\letters\letter_س.mp3



number 0 -> C:\Users\sondo\Desktop\Voice Model Stuff\audio\numbers\number_0.mp3



number (cut) 2 -> C:\Users\sondo\Desktop\Voice Model Stuff\audio\numbers\number_2.mp3



number 10 -> C:\Users\sondo\Desktop\Voice Model Stuff\audio\numbers\number_10.mp3
